In [1]:
import pandas as pd

# Step 1: Load the dataset
file_path = 'E:\SignalModel\Out_7.csv'
df = pd.read_csv(file_path)

# Step 2: Ensure 'S' column is in datetime format
df['datetime'] = pd.to_datetime(df['datetime'])

# Step 3: Define the groups
group1 = ['SS', 'MS', 'WS']
group2 = ['SB', 'MB', 'WB']

# Function to determine the signal based on the columns
def get_signal(row):
    for signal in group1 + group2:
        if row[signal] == 1:
            return signal
    return None

# Step 4: Create buckets based on the sequence of signals
buckets = []
current_bucket = {'Dates': [], 'Values': [], 'Signal': [], 'Count': 0}
current_group = None

for i in range(len(df)):
    signal = get_signal(df.iloc[i])
    
    if signal is None:
        continue
    
    group = group1 if signal in group1 else group2
    
    if current_group is None:
        current_group = group

    if group == current_group:
        current_bucket['Dates'].append(df.at[i, 'datetime'])
        current_bucket['Values'].append(df.at[i, 'sigai_output'])
        current_bucket['Signal'].append(signal)
        current_bucket['Count'] += 1
    else:
        if current_bucket['Count'] > 0:
            buckets.append(current_bucket)
        current_bucket = {'Dates': [df.at[i, 'datetime']], 'Values': [df.at[i, 'sigai_output']], 'Signal': [signal], 'Count': 1}
        current_group = group

# Append the last bucket if not empty
if current_bucket['Count'] > 0:
    buckets.append(current_bucket)

# Step 5: Create the Output DataFrame
output_data = {
    'Bucket': [],
    'Dates': [],
    'Values': [],
    'Signal': [],
    'Count': []
}

for idx, bucket in enumerate(buckets, start=1):
    output_data['Bucket'].append(idx)
    output_data['Dates'].append(bucket['Dates'])
    output_data['Values'].append(bucket['Values'])
    output_data['Signal'].append(bucket['Signal'])
    output_data['Count'].append(bucket['Count'])

output_df = pd.DataFrame(output_data)

# Step 6: Save the DataFrame to a CSV file
output_file_path = 'E:\SignalModel\Sell and Buy Bucketing\Bucketed_Output.csv'
output_df.to_csv(output_file_path, index=False)


In [1]:
import pandas as pd
import re

# Step 1: Load the datasets
bucketed_file_path = 'E:\SignalModel\Sell and Buy Bucketing\Bucketed_Output.csv'
zigzag_file_path = 'E:\SignalModel\SS-SB\ZigZag_Output\zigzag_output_0.019.csv'

bucketed_df = pd.read_csv(bucketed_file_path)
zigzag_df = pd.read_csv(zigzag_file_path)

# Ensure date columns are in datetime format
zigzag_df['datetime'] = pd.to_datetime(zigzag_df['datetime'])

# Extract local minima and maxima dates
local_min_dates = set(zigzag_df[zigzag_df['local_min'] == 1]['datetime'])
local_max_dates = set(zigzag_df[zigzag_df['local_max'] == 1]['datetime'])

# Function to parse lists of dates in 'Dates' column
def parse_dates(dates_str):
    # Remove the Timestamp('...') wrapper
    dates_list_str = re.findall(r"Timestamp\('(.*?)'\)", dates_str)
    # Convert each string to a pd.Timestamp
    return [pd.to_datetime(date_str) for date_str in dates_list_str]

# Parse the 'Dates' column in bucketed_df
bucketed_df['Dates'] = bucketed_df['Dates'].apply(parse_dates)

# Step 3: Add local extrema information to the bucketed DataFrame
def get_local_extrema(dates):
    extrema_types = []
    for date in dates:
        if date in local_min_dates:
            extrema_types.append('local_min')
        if date in local_max_dates:
            extrema_types.append('local_max')
    return extrema_types

bucketed_df['Local Extrema'] = bucketed_df['Dates'].apply(get_local_extrema)
# Add "Sell\Buy" column with alternating values
bucketed_df['Sell\\Buy'] = ['Sell' if i % 2 == 0 else 'Buy' for i in range(len(bucketed_df))]

In [2]:
# Function to get local extrema types
def get_local_extrema(dates):
    extrema_types = []
    for date in dates:
        if date in local_min_dates:
            extrema_types.append('local_min')
        if date in local_max_dates:
            extrema_types.append('local_max')
    return extrema_types

# Add 'Local Extrema' column to bucketed_df
bucketed_df['Local Extrema'] = bucketed_df['Dates'].apply(get_local_extrema)

# Initialize new columns for counting local extrema based on sell and buy signals
bucketed_df['Sell Local Min Count'] = 0
bucketed_df['Sell Local Max Count'] = 0
bucketed_df['Buy Local Min Count'] = 0
bucketed_df['Buy Local Max Count'] = 0

# Function to count local extrema based on 'Sell\Buy' column
def count_local_extrema(dates, local_extrema, signal):
    local_min_count = sum(1 for ext in local_extrema if ext == 'local_min')
    local_max_count = sum(1 for ext in local_extrema if ext == 'local_max')
    return local_min_count, local_max_count

# Apply the function to count local extrema based on sell and buy signals
for idx, row in bucketed_df.iterrows():
    dates = row['Dates']
    signal = row['Sell\\Buy']
    local_extrema = row['Local Extrema']
    
    if signal == 'Sell':
        sell_local_min_count, sell_local_max_count = count_local_extrema(dates, local_extrema, signal)
        bucketed_df.at[idx, 'Sell Local Min Count'] = sell_local_min_count
        bucketed_df.at[idx, 'Sell Local Max Count'] = sell_local_max_count
    elif signal == 'Buy':
        buy_local_min_count, buy_local_max_count = count_local_extrema(dates, local_extrema, signal)
        bucketed_df.at[idx, 'Buy Local Min Count'] = buy_local_min_count
        bucketed_df.at[idx, 'Buy Local Max Count'] = buy_local_max_count

# Save the updated DataFrame to a CSV file
updated_output_file_path = 'E:\SignalModel\Sell and Buy Bucketing\\Updated_Bucketed_Output_0.019.csv'
bucketed_df.to_csv(updated_output_file_path, index=False)
